# load package

In [ ]:
import numpy as np
import pandas as pd
import random as rd
import seaborn as sns
from scipy import interp
import scipy.stats as st
import os
import copy
import re
import time
from copy import deepcopy
import matplotlib.pyplot as plt
import cloudpickle as pickle
from scipy import stats
from sklearn.metrics import roc_curve, auc,accuracy_score, precision_score, recall_score,f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Lasso
from sklearn import *
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
import warnings
warnings.filterwarnings('ignore')

# function definition

In [ ]:
MLA = [
    #Ensemble Methods
    ensemble.GradientBoostingClassifier(),
    ensemble.RandomForestClassifier(),
    #GLM
    linear_model.LogisticRegression(),
    #NN
    neural_network.MLPClassifier()
]

# Intra-population cross-validation

In [ ]:
def get_self_cv(alg, data, Groups, k=5, SEED = 5):
    splitor = StratifiedKFold(n_splits=k, shuffle=True,random_state=SEED)
    ### alg choice
    if alg == 'gbc':
        clf = GradientBoostingClassifier(n_estimators=101, learning_rate=0.1, subsample=1,loss='deviance',
                                         max_features=0.1, min_samples_leaf=1, random_state=SEED,
                                         criterion='friedman_mse',max_depth=3)
    if alg == 'rfc':
        clf = RandomForestClassifier(n_estimators = 501, oob_score = True, random_state =SEED,class_weight='balanced', 
                                max_features = 'log2', criterion='gini', n_jobs = 2) ###max_depth=5,min_samples_leaf=10,class_weight='balanced'
    if alg == 'lrcv':
        clf = LogisticRegression(penalty='l2', dual=False, C=1.0, fit_intercept=True, intercept_scaling=1,
                                 class_weight=None, random_state=SEED, solver='liblinear', 
                                 max_iter=100, multi_class='ovr', verbose=0, warm_start=False, n_jobs=1)
    if alg == 'mlpc':
        clf = MLPClassifier(solver='lbfgs', alpha=1e-5,hidden_layer_sizes=(10,5), random_state=SEED) 

    aucs = []
    accs = []
    percisions = []
    recalls = []
    F1s = []
    i = 0
    for train_index, test_index in splitor.split(data, Groups):
        y_train, y_test = Groups[train_index], Groups[test_index]
        X_train, X_test = np.array(data)[train_index], np.array(data)[test_index]
        pred = clf.fit(X_train, y_train).predict(X_test)
        probas = clf.fit(X_train, y_train).predict_proba(X_test)
        fpr, tpr, thresholds = roc_curve(y_test, probas[:, 1])
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)
        acc = accuracy_score(y_test, pred)
        accs.append(acc)
        percision = precision_score(y_test, pred)
        percisions.append(percision)
        recall = recall_score(y_test, pred)
        recalls.append(recall)
        f1score = f1_score(y_test,pred)
        F1s.append(f1score)
        i += 1
    Auc = np.mean(aucs)
    Recall = np.mean(recalls)
    Precision = np.mean(percisions)
    F1_score = np.mean(F1s)
    Accuracy = np.mean(accs)
    return Auc, Recall, Precision, F1_score, Accuracy,clf 
      

# Data Preparation

In [ ]:
qval=pd.read_csv("CRC_fungi_pvalue_block_CHIJAP_abunt_test.csv",index_col=0)
diff=list(qval.loc[qval['all']<0.01,:].index)
print(diff)
print(len(diff))
CommonID=diff

In [ ]:
data_feat=pd.read_csv("NewProfile/CRC_fungi_merged_species_rela.csv",index_col=0)
data_feat_all=data_feat.T[CommonID]
data_feat_all.shape

In [ ]:
metadata=pd.read_csv("metadata/CRC_count_metadata.csv",index_col=1)

meta_all=metadata.loc[(metadata['Group']!='adenoma')& (metadata['Group']!='HS'),['Run','Study','Group','BMI','Age']]
meta_cn=metadata.loc[(metadata['Dataset']=='Discovery')& (metadata['Group']!='adenoma')& 
                     (metadata['Group']!='HS'),['Run','Study','Group','BMI','Age']]
meta_cn.head()
meta_cn.shape

In [ ]:
data_cn = pd.merge(meta_cn,data_feat_all,left_on='Run',right_index=True)
print(data_cn.shape)
data_cn.head()

In [ ]:
site=list(set(data_cn['Study']))
site.sort()
print(site)

# The whole data model and feature importance

In [ ]:
for s in site:
    print(s)
    meta_study=meta_cn.loc[meta_cn['Study']==s,:]
    group=meta_study['Group']

    labelCN=group.replace({'CTR':0,'CRC':1})
    labelCN.index=list(meta_study.Run)
    labelCN.columns = ['Group']
    labelCN=labelCN.to_frame()
    #print(type(labelCN))
    label = np.array([i for i in labelCN.Group])
    print(label)
    data_study=data_cn.loc[data_cn['Study']==s,:]
    Auc, Recall, Precision, F1_score, Accuracy, clf = get_self_cv(alg='rfc', data=data_study.loc[:,CommonID], Groups=label, k=5, SEED = 5)
    print(Auc)
    Feature_rank = sorted(zip(map(lambda x: round(x, 4), clf.feature_importances_), CommonID), reverse=True)
    print("Features sorted by their score:")
    print(Feature_rank)
    fileout=open('temp/CRC_Fungi_feature_importance_'+s+'.txt','w')
    for i in range(len(CommonID)):
        fileout.write(Feature_rank[i][1]+'\t'+str(Feature_rank[i][0])+'\n')
    fileout.close()
print('Done')

# Rank aggregation in R

In [ ]:
Rscript feature_rank_aggregation.R

# Feature select according to the aggragated importance

In [ ]:
Rankfile=pd.read_csv("temp/CRC_fungi_feature_importance_RankAgg.csv",index_col=0)
Importance_rank=list(Rankfile['rank'])
print(Importance_rank)

In [ ]:
Auc_columns = ['Feature','AUS','CHI','FRA','GER','JAP','mean','sd']
Auc_report = pd.DataFrame(columns = Auc_columns)
for i in range(1,(len(Importance_rank)+1)):
    aucs=[]
    feature=[]
    for m in range(i):
        feature.append(Importance_rank[m])
    
    for s in site:
        #random_columns = ['AUC','Recall','Precision','F1','accuracy']
        #random_results = pd.DataFrame(columns = random_columns)
        data = data_cn.loc[data_cn['Study'] == s,]
        data_feat = data.loc[:,feature]
        group = data['Group']
        label = group.replace({'CTR':0,'CRC':1})
        label.columns = ['Group']
        label = label.to_frame()
        label = np.array([n for n in label.Group])
        Auc, Recall, Precision, F1_score, Accuracy,clf  = get_self_cv(alg='rfc', data=data_feat,Groups=label, k=10, SEED = 5) 
        aucs.append(Auc)
        Auc_report.loc[i,s]=Auc
    mean_auc = np.mean(aucs)
    max_auc = np.max(aucs)
    min_auc = np.min(aucs)
    sd_auc = np.var(aucs)
    Auc_report.loc[i,['Feature','mean','sd']]=[feature,mean_auc,sd_auc]
    #Auc_report.append([feature, mean_auc,max_auc,min_auc])
    print(len(feature),mean_auc,max_auc,min_auc)

In [ ]:
pickle.dump(Auc_report, open('temp/CRC_Fungi_feature_select_CHIJAP_01.pkl','wb'))

In [ ]:
Auc_report.to_csv('temp/CRC_Fungi_feature_select_CHIJAP.csv')